# Poisoned Chalice Competition 2026 - Kaggle Starter

This notebook provides a starter environment to run the Membership Inference Attacks (MIA) for the Poisoned Chalice Competition.

## 1. Setup Environment
Clone the repository and install dependencies.

In [ ]:
!git clone https://github.com/technoob05/PoisonedChalice.git
%cd PoisonedChalice
# Install specific dependencies to avoid OS-specific issues with requirements.txt
!pip install transformers datasets accelerate scikit-learn pandas numpy matplotlib huggingface_hub

## 2. Hugging Face Login
We use Kaggle Secrets to log in to Hugging Face automatically. Ensure you have added your HF Token as a secret named `posioned`.

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("posioned")
    login(token=secret_value_0)
    print("Successfully logged in via Kaggle Secrets.")
except Exception as e:
    print(f"Could not log in via secrets: {e}")
    print("Falling back to interactive login...")
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Run Experiment
Run the `run.py` script. 
- Choose a model (e.g., `bigcode/starcoder2-3b`).
- Select attacks (`loss`, `mkp`, `pac`).
- Adjust `sample_fraction` for testing (use 1.0 for full run).
- **Dataset**: Using the uploaded Kaggle dataset at `/kaggle/input/datasets/minh2duy/poisoned-chalice-dataset`.

In [ ]:
# Example run with a small sample fraction for testing
!python run.py \
    --model_name "bigcode/starcoder2-3b" \
    --attacks loss mkp \
    --sample_fraction 0.01 \
    --output_dir results \
    --dataset "/kaggle/input/datasets/minh2duy/poisoned-chalice-dataset" \
    --use_fp16

## 4. Process Results
Calculate metrics and generate plots (ROC Curves).

In [ ]:
import glob
import os

# Create plots directory
os.makedirs("plots", exist_ok=True)

# Find latest metadata file
metadata_files = glob.glob("results/metadata_*.json")
if metadata_files:
    latest_metadata = max(metadata_files, key=os.path.getctime)
    print(f"Processing results for: {latest_metadata}")
    !python process.py --config_path "{latest_metadata}" --results_folder results/ --output_path plots/
else:
    print("No metadata file found. run.py might have failed.")

## 5. View Results

In [ ]:
import pandas as pd
import glob
import os

# Find the latest results file
result_files = glob.glob("results/*.parquet")
if result_files:
    latest_result = max(result_files, key=os.path.getctime)
    df = pd.read_parquet(latest_result)
    display(df.head())
    print(f"Loaded results from {latest_result}")

    # Interpreting scores:
    # loss_score: Cross Entropy Loss. Lower is better (more likely member).
    # mkp_score: Min-K % Probability. Higher is better (more likely member).
else:
    print("No result files found.")